In [ ]:

import os
import subprocess
import gc

import polars as pl
from phetk.phecode import Phecode
from phetk.phewas import PheWAS


bucket_name = os.getenv("WORKSPACE_BUCKET", "").replace("gs://", "").strip("/")
path_base   = f"gs://{bucket_name}/sjl"
path_inv    = f"{path_base}/invariant"
path_out    = f"{path_base}/outputs"

PHECODE_VERSION      = "X"
ICD_VERSION          = "US"
MIN_CASES            = 50
MIN_PHECODE_COUNT    = 2
RESTRICT_TO_EMPLOYED = True
MALE_AS_ONE          = True

FDR_HIT_THRESHOLD = 0.05   


COVARS_MINIMAL = [
    "age_at_landmark",
    "sex",
    "deprivation_index",
    "ehr_length_years",
]

COVARS_PRIMARY = COVARS_MINIMAL + [
    "median_daily_steps",
    "avg_sleep_duration_hours",
    "smoke_smoke_ever",
    "alcohol_alcohol_occasional",
    "alcohol_alcohol_regular",
    "edu_edu_hs",
    "edu_edu_college_plus",
    "income_income_mid",
    "income_income_high",
]

EMP_STRUCTURED_COL = "emp_emp_structured"
EXPOSURE           = "avg_sjl_hours"
SEX_COL            = "sex"

def gcs_exists(path: str) -> bool:
    return subprocess.call(["gsutil", "-q", "stat", path]) == 0


def local_copy(gcs_path: str, local_path: str) -> str:
    subprocess.call(["gsutil", "cp", gcs_path, local_path])
    return local_path


def upload(local_path: str, gcs_path: str) -> None:
    subprocess.call(["gsutil", "cp", local_path, gcs_path])
    print(f"Uploaded: {gcs_path}")


def tsv_to_parquet(tsv_path: str, parquet_path: str) -> None:
    df = pl.read_csv(tsv_path, separator="\t", try_parse_dates=True,
                     infer_schema_length=10_000)
    df.write_parquet(parquet_path)
    print(f"Converted TSV → parquet: {parquet_path}")


def bh_fdr(p_series: pl.Series) -> pl.Series:
    """Benjamini-Hochberg FDR on a pl.Series. Returns adjusted q-values."""
    n        = len(p_series)
    order    = p_series.arg_sort()
    ranks    = order.arg_sort() + 1
    bh_q     = (p_series * n / ranks).clip(upper_bound=1.0)
    sorted_q = bh_q.gather(order).reverse().cum_min().reverse()
    return sorted_q.gather(order.arg_sort())


print("Cleaning up stale PheWAS outputs from previous runs...")
subprocess.call(
    ["gsutil", "-m", "rm", "-f",
     f"{path_out}/phewas_logistic_primary.parquet",
     f"{path_out}/phewas_logistic_minimal.parquet",
     f"{path_out}/logistic_hit_phecodes.csv"],
    stderr=subprocess.DEVNULL,
)
print("  Done.")

print("\nDownloading invariant files...")
local_phecode = local_copy(f"{path_inv}/phecode_counts.tsv",
                           "/tmp/phecode_counts.tsv")
local_cohort  = local_copy(f"{path_inv}/cohort_with_landmark.tsv",
                           "/tmp/cohort_with_landmark.tsv")

if RESTRICT_TO_EMPLOYED:
    print("Restricting cohort to emp_structured == 1...")
    cohort_df = pl.read_csv(local_cohort, separator="\t", try_parse_dates=True,
                            infer_schema_length=10_000)

    if EMP_STRUCTURED_COL not in cohort_df.columns:
        raise KeyError(
            f"FATAL: '{EMP_STRUCTURED_COL}' not found in cohort TSV. "
            f"Re-run sjl_pipeline.py Step 14. "
            f"Employment columns present: "
            f"{[c for c in cohort_df.columns if 'emp' in c.lower()]}"
        )

    cohort_df = cohort_df.filter(pl.col(EMP_STRUCTURED_COL) == 1)
    local_cohort_employed = "/tmp/cohort_employed.tsv"
    cohort_df.write_csv(local_cohort_employed, separator="\t")
    print(f"  Restricted cohort: {cohort_df.height:,} participants")
    COHORT_PATH = local_cohort_employed
    del cohort_df
    gc.collect()
else:
    COHORT_PATH = local_cohort

_head      = pl.read_csv(COHORT_PATH, separator="\t", n_rows=1, infer_schema_length=1)
_available = set(_head.columns)
_expected  = set(COVARS_PRIMARY + [EMP_STRUCTURED_COL])
_missing   = _expected - _available

if _missing:
    raise KeyError(
        f"FATAL: Missing covariate columns from cohort_with_landmark.tsv:\n"
        f"  {sorted(_missing)}\n"
        f"Re-run sjl_pipeline.py Step 14."
    )

print(f"\nAll covariate columns verified ({len(_expected)} checked).")
print(f"Primary  ({len(COVARS_PRIMARY)}): {COVARS_PRIMARY}")
print(f"Minimal  ({len(COVARS_MINIMAL)}): {COVARS_MINIMAL}")

def run_logistic(label: str, covars: list[str]) -> pl.DataFrame:
    """
    Run one full-scan logistic PheWAS via phetk.
    Applies convergence filter, FDR correction, Bonferroni flag.
    Uploads parquet to GCS and returns the processed DataFrame.
    """
    print(f"\n{'='*60}")
    print(f"LOGISTIC PHEWAS (prevalent) — {label}")
    print("="*60)

    out_tsv     = f"/tmp/phewas_logistic_{label}.tsv"
    out_parquet = f"/tmp/phewas_logistic_{label}.parquet"

    phewas = PheWAS(
        phecode_version                  = PHECODE_VERSION,
        phecode_count_file_path          = local_phecode,
        cohort_file_path                 = COHORT_PATH,
        covariate_cols                   = covars,
        independent_variable_of_interest = EXPOSURE,
        sex_at_birth_col                 = SEX_COL,
        male_as_one                      = MALE_AS_ONE,
        method                           = "logit",
        icd_version                      = ICD_VERSION,
        min_cases                        = MIN_CASES,
        min_phecode_count                = MIN_PHECODE_COUNT,
        use_exclusion                    = False,
        output_file_path                 = out_tsv,
        suppress_warnings                = True,
        verbose                          = False,
        batch_size                       = None,
        fall_back_to_serial              = True,
    )
    phewas.run(parallelization="multithreading")
    del phewas
    gc.collect()

    if not os.path.exists(out_tsv):
        raise RuntimeError(
            f"FATAL: phetk did not produce {out_tsv}. "
            f"Check phetk output above for errors."
        )

    tsv_to_parquet(out_tsv, out_parquet)
    df = pl.read_parquet(out_parquet)
    n_raw = len(df)

    # Convergence filter
    df = df.filter(pl.col("converged") == True)
    n_converged = len(df)

    # Stability filter
    df = df.filter(pl.col("beta").abs() <= 10)
    n_stable = len(df)

    print(f"  Raw results:       {n_raw:,}")
    print(f"  After convergence: {n_converged:,} (dropped {n_raw - n_converged:,})")
    print(f"  After |beta|<=10:  {n_stable:,} (dropped {n_converged - n_stable:,})")

    n_tests = n_raw
    p_series = df["p_value"]
    order    = p_series.arg_sort()
    ranks    = order.arg_sort() + 1
    bh_q     = (p_series * n_tests / ranks).clip(upper_bound=1.0)
    sorted_q = bh_q.gather(order).reverse().cum_min().reverse()
    fdr_q    = sorted_q.gather(order.arg_sort())
    df = df.with_columns(pl.Series("fdr_q", fdr_q))

    rename_map = {k: v for k, v in {
        "p_value":        "p_val",
        "standard_error": "std_error",
        "cases":          "n_cases",
        "controls":       "n_controls",
    }.items() if k in df.columns}
    df = df.rename(rename_map)

    df = df.with_columns(
        (pl.col("p_val") < 0.05 / n_tests).alias("bonf_sig")
    )

    df.write_parquet(out_parquet)
    upload(out_parquet, f"{path_out}/phewas_logistic_{label}.parquet")

    n_fdr  = (df["fdr_q"] < FDR_HIT_THRESHOLD).sum()
    n_bonf = df["bonf_sig"].sum()
    print(f"  FDR < {FDR_HIT_THRESHOLD}:       {n_fdr:,}")
    print(f"  Bonferroni < 0.05: {n_bonf:,}")

    return df

logistic_primary = run_logistic("primary", COVARS_PRIMARY)


hits = (
    logistic_primary
    .filter(pl.col("fdr_q") < FDR_HIT_THRESHOLD)
    .select([
        "phecode", "phecode_string", "phecode_category",
        "beta", "std_error", "p_val", "fdr_q", "bonf_sig",
        "odds_ratio",
        "n_cases", "n_controls",
    ] + (["odds_ratio_low", "odds_ratio_high"]
         if "odds_ratio_low" in logistic_primary.columns else []))
    .sort("p_val")
)

hit_csv = "/tmp/logistic_hit_phecodes.csv"
hits.write_csv(hit_csv)
upload(hit_csv, f"{path_out}/logistic_hit_phecodes.csv")

n_hits = len(hits)
print(f"\nExported {n_hits:,} hit phecodes (FDR < {FDR_HIT_THRESHOLD}) to:")
print(f"  {path_out}/logistic_hit_phecodes.csv")

del logistic_primary
gc.collect()

logistic_minimal = run_logistic("minimal", COVARS_MINIMAL)
del logistic_minimal
gc.collect()

print("\n=== PHEWAS NOTEBOOK COMPLETE ===")
print(f"\nOutputs → {path_out}")
print("  phewas_logistic_primary.parquet   ← full-scan discovery (primary covariates)")
print("  phewas_logistic_minimal.parquet   ← full-scan discovery (minimal, appendix)")
print(f"  logistic_hit_phecodes.csv         ← {n_hits} FDR hits → R Cox validation")
print("\nNext: run sjl_analysis.R")
print("  Loads logistic_hit_phecodes.csv and phewas_logistic_*.parquet")
print("  Runs targeted Cox (primary / sens_bmi / sens_apnea) for each hit phecode")
print("  Produces all main and supplementary figures and tables")